# TP01 - Première utilisation des données

In [2]:
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import numpy as np

file = "../data/food.parquet"

batch_size = 70000

### 3. Premier dataset et caractéristiques

#### Chargement du dataset complet

In [ ]:
#Chargement complet avec Pandas (à éviter)
fichier_charge = pq.ParquetFile(file)
dt = pd.DataFrame()
for batch in fichier_charge.iter_batches(batch_size=batch_size):
    temp_df = batch.to_pandas()
    dt = pd.concat([dt,temp_df])
len(dt)

In [ ]:
# Affichage des colonnes sans charger le fichier
rel = duckdb.sql("SELECT * FROM '../data/food.parquet' limit 0")
rel.columns

#### Caractéristiques :

- Dimensions : (4636471, 111)
- Types : bool(1), float32(1), float64(25), int32(1), object(56), str(27)
- Utilisation mémoire : 21.8+ GB de ram
- Taux de remplissage par colonne : voir le code

In [ ]:
print(f"Dimensions : {dt.shape}")
print(f"Informations globales : \n")
dt.info()
print(f"Taux de remplissage par colonne :")
dt.notna().mean()*100

### 4. Cinq questions à résoudre

- combien de produits vendus en France ?
- quelle part a un Nutri-Score renseigné ?
- les dix marques les plus présentes ?
- le taux de manquants sur les nutriments clés ( energy_100g ,sugars_100g , salt_100g ) ?
- qu'est-ce qui vous semble le plus « sale » dans ces données ?

#### Chargement d'un dataset filtré

> Pour répondre à ces questions, on a fait le choix de sélectionner une version filtré du dataset, avec seulement les colonnes qui nous intéressent.

Chargement des données avec PyArrow

In [3]:
fichier_charge = pq.ParquetFile(file)
df_france = pd.DataFrame()
for batch in fichier_charge.iter_batches(batch_size=batch_size, columns=["product_name", "categories", "brands", "countries_tags", "nutriscore_score", "nutriscore_grade", "nutriments"]):
    temp_df = batch.to_pandas()
    temp_df = temp_df[temp_df["countries_tags"].apply(lambda x : isinstance(x, np.ndarray) and "en:france" in x)]
    df_france = pd.concat([df_france,temp_df])
print(df_france.memory_usage(deep=True))

Index                 9978688
product_name        149680320
categories           72371109
brands               17501043
countries_tags      149680320
nutriscore_score      9978688
nutriscore_grade     16194759
nutriments          119952672
dtype: int64


Chargement des données avec DuckDB

In [ ]:
#Chargement avec duckdb
df_france = duckdb.sql("""
    SELECT "product_name", "categories", "brands", "countries_tags", "nutriscore_score", "nutriscore_grade", "nutriments"
    FROM '../data/food.parquet'
    WHERE list_contains(countries_tags, 'en:france')
""").df()

Chargement des données avec Pandas

In [ ]:
#Chargement avec Pandas
dt = pd.read_parquet(file, columns=["product_name", "categories", "brands", "countries_tags", "nutriscore_score", "nutriscore_grade", "nutriments"])
df_france = dt[dt["countries_tags"].apply(lambda x : isinstance(x, np.ndarray) and "en:france" in x)]
len(df_france)

Chargement des données avec Polars

### Optimisation du dataset

Colonnes à optimiser :
- nutriscore_grade : score entre a et e
- nutriscore_score : score entre -17 et 57
- categories : 120 000 catégories sur 1 400 000 lignes

In [ ]:
df_france.groupby("categories")["categories"].count().sort_values(ascending=False)

In [18]:
print(df_france["nutriscore_grade"].memory_usage(deep=True))
df_france["nutriscore_grade"] = df_france["nutriscore_grade"].astype("category")
print(df_france["nutriscore_grade"].memory_usage(deep=True))

11226107
11226107


In [17]:
#Nutriscore en int8
print(df_france["nutriscore_score"].memory_usage(deep=True))
df_france["nutriscore_score"] = df_france['nutriscore_score'].astype("Int8")
print(df_france["nutriscore_score"].memory_usage(deep=True))

12473360
12473360


Récupération des éléments ayant un nutriscore renseigné

In [ ]:
dt.count()
#dt[dt["nutriscore_grade"].isin(["a","b","c","d","e"])]["nutriscore_grade"].count()
dt.groupby("nutriscore_grade")["nutriscore_grade"].count()

le taux de manquants sur les nutriments clés ( energy_100g , sugars_100g , salt_100g )

In [ ]:
#dt = pd.read_parquet(file, columns=["product_name", "countries_tags"])

In [ ]:
#dt[dt['countries_tags'] == ["en:france"]]

In [ ]:
rel = duckdb.sql("SELECT f.product_name, f.nutriments FROM '../data/food.parquet' as f limit 10").df()
rel